# IronStore Enterprise AI Assistant — v2 (Pinecone **hybrid** search)

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

**What changed from v1** (`ironstore_assistant_v1.ipynb` stays untouched):

| | v1 | 🆕 v2 |
|---|---|---|
| Search | dense only (cosine, *meaning*) | **hybrid** = dense (*meaning*) + sparse (*exact words & codes*) |
| Keyword model | — | Pinecone hosted `pinecone-sparse-english-v0` (no install) |
| Pinecone index | `ironstore-assistant` (cosine) | `ironstore-hybrid` (dotproduct — required for hybrid) |
| Mix of both | — | **α** (alpha): 1 = meaning only, 0 = keywords only; chosen automatically per question |
| SKU question | ❌ "couldn't find" | ✅ expected to work |

**Legend:** <span style="background:#e8f5e9;border-left:4px solid #2e7d32;padding:0 6px">🆕 green = new in v2</span> &nbsp;
<span style="background:#f3f3f3;border-left:4px solid #9e9e9e;padding:0 6px">♻️ grey = same as v1</span>
</div>


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Step 0 — Speed fix (this PC only) <span style="color:#757575;font-size:0.8em">(same as v1)</span>

Try IPv4 before IPv6 so each API call takes ~1 s instead of ~40 s.

</div>

In [1]:
import socket

if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 1 — Settings

Same as v1, plus a **new index name** and the **sparse (keyword) model**.

</div>

In [2]:
import os
import re
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()

DOCS_DIR = Path(r"C:\Users\Windows\Desktop\Ironhack\04_llm\project-brief-enterprise-ai-assistant\documents\extracted\documents")
INDEX_NAME = "ironstore-hybrid"                 # 🆕 v2 (v1: ironstore-assistant)
EMBED_MODEL = "text-embedding-3-small"          # dense: meaning
EMBED_DIM = 1536
SPARSE_MODEL = "pinecone-sparse-english-v0"     # 🆕 v2 sparse: exact words, hosted by Pinecone
REWRITE_MODEL = "gpt-4o-mini"
ANSWER_MODEL = "gpt-4.1-mini"

print("OpenAI key found:  ", bool(os.getenv("OPENAI_API_KEY")))
print("Pinecone key found:", bool(os.getenv("PINECONE_API_KEY")))

OpenAI key found:   True
Pinecone key found: True


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Steps 2–4 — Load the PDFs and split into chunks <span style="color:#757575;font-size:0.8em">(same as v1)</span>

83 PDFs → one record per document (area, title, page, last reviewed, outdated) → ~450-token chunks with a context header.

</div>

In [3]:
from bisect import bisect_right
import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf(path: Path) -> dict:
    pages = [page.get_text() for page in pymupdf.open(path)]
    text = "".join(pages)
    head = text[:600]
    match = re.search(r"Last reviewed(?: in|:)?\s*([A-Z][a-z]+ \d{4})", head)
    last_reviewed = match.group(1) if match else "unknown"
    year = int(last_reviewed[-4:]) if match else 0
    if path.parent.name == "product_catalog":
        title = path.stem.replace("IronStore_", "").replace("_", " ")
    else:
        title = head.strip().splitlines()[0]
    page_starts, pos = [], 0
    for page in pages:
        page_starts.append(pos)
        pos += len(page)
    return {"path": path, "file": path.name, "area": path.parent.name,
            "doc_type": "catalog" if path.parent.name == "product_catalog" else "policy",
            "title": title, "last_reviewed": last_reviewed, "outdated": 0 < year < 2026,
            "text": text, "page_starts": page_starts}

docs = [read_pdf(p) for p in sorted(DOCS_DIR.rglob("*.pdf"))]

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=450, chunk_overlap=60)
chunks = []
for doc in docs:
    start = 0
    for n, piece in enumerate(splitter.split_documents([Document(page_content=doc["text"])])):
        start = max(doc["text"].find(piece.page_content, start), start)
        page = bisect_right(doc["page_starts"], start)
        header = (f"[{doc['title']} | {doc['area']} | page {page} | last reviewed {doc['last_reviewed']}"
                  + (" | OUTDATED" if doc["outdated"] else "") + "]")
        chunks.append({"id": f"{doc['path'].stem}-{n:03d}", "text": header + "\n" + piece.page_content,
                       "source": doc["file"], "title": doc["title"], "area": doc["area"],
                       "doc_type": doc["doc_type"], "page": page, "last_reviewed": doc["last_reviewed"],
                       "outdated": doc["outdated"], "path": str(doc["path"])})

print(len(docs), "documents →", len(chunks), "chunks")

83 documents → 1759 chunks


<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 5 — A hybrid Pinecone index

Pinecone stores **two vectors per chunk** in one record:
- 🔵 **dense** `values` — 1536 numbers from OpenAI (meaning)
- 🟠 **sparse** `sparse_values` — only the words that occur, each with a weight (like keyword search)

Hybrid indexes need the metric **`dotproduct`** (v1 used `cosine`). OpenAI vectors all have length 1, so for the dense part *dotproduct = cosine* — nothing is lost.

</div>

In [4]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME, dimension=EMBED_DIM, metric="dotproduct",   # 🆕 v2: dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("index created")
index = pc.Index(host=pc.describe_index(INDEX_NAME).host)
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=1759, metric='dotproduct', namespaces=1)


<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 6 — Look at a sparse vector

Pinecone's keyword model turns text into **tokens with weights**. See how the product code is kept as searchable pieces (`prj`, `pcb`, `pb1080`) — this is exactly what the dense vector can't do.

</div>

In [5]:
def sparse_embed(texts, input_type):
    # 🆕 v2: keyword vectors from Pinecone's hosted model, 96 texts per call (with a retry on rate limits)
    out = []
    for i in range(0, len(texts), 96):
        for attempt in range(5):
            try:
                result = pc.inference.embed(model=SPARSE_MODEL, inputs=texts[i:i + 96],
                                            parameters={"input_type": input_type, "truncate": "END"})
                break
            except Exception as error:
                if "429" not in str(error) or attempt == 4:
                    raise
                time.sleep(20)
        out += [{"indices": e.sparse_indices, "values": e.sparse_values} for e in result.data]
    return out

example = pc.inference.embed(model=SPARSE_MODEL, inputs=["What is SKU PRJ-PCB-PB1080 and how much does it cost?"],
                             parameters={"input_type": "query", "return_tokens": True}).data[0]
print("tokens:", example.sparse_tokens)

tokens: ['cost', 'what', 'pcb', 'prj', 'pb1080', 'is', 'it', 'sku', 'and', 'does', 'how', 'much']


<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 7 — Embed (dense + sparse) and upload

Same as v1, plus a sparse vector for every chunk. Skipped when the index is already full (set `REBUILD = True` after changing the chunks).

</div>

In [6]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
REBUILD = False

already = index.describe_index_stats().total_vector_count
if REBUILD or already != len(chunks):
    if already:
        index.delete(delete_all=True)
    texts = [c["text"] for c in chunks]
    t0 = time.time()
    dense = embeddings.embed_documents(texts)
    print(f"dense:  {len(dense)} vectors in {time.time() - t0:.0f} s")
    t0 = time.time()
    sparse = sparse_embed(texts, "passage")
    print(f"sparse: {len(sparse)} vectors in {time.time() - t0:.0f} s")
    index.upsert(
        vectors=[{"id": c["id"], "values": d, "sparse_values": s, "metadata": c}
                 for c, d, s in zip(chunks, dense, sparse)],
        batch_size=100, show_progress=False,
    )
    while index.describe_index_stats().total_vector_count < len(chunks):
        time.sleep(2)
else:
    print("Pinecone already has all", already, "chunks — nothing to upload")
print(index.describe_index_stats())

Pinecone already has all 1759 chunks — nothing to upload
DescribeIndexStatsResponse(dimension=1536, total_vector_count=1759, metric='dotproduct', namespaces=1)


<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 8 — Hybrid search with α (alpha)

One query sends **both** vectors. α decides the mix:

`dense × α` + `sparse × (1 − α)` → **α = 1** meaning only (= v1) · **α = 0** keywords only · **α = 0.5** both equally

The table compares where the **right chunk** lands for a SKU question and for a normal question.

</div>

In [7]:
def search(dense_vec, sparse_vec, alpha=0.5, k=6):
    alpha = min(max(alpha, 0.01), 1.0)        # Pinecone needs a non-zero dense vector
    result = index.query(
        vector=[v * alpha for v in dense_vec],
        sparse_vector={"indices": sparse_vec["indices"], "values": [v * (1 - alpha) for v in sparse_vec["values"]]},
        top_k=k, include_metadata=True,
    )
    return [{"id": m.id, "score": round(m.score, 3), **m.metadata} for m in result.matches]

def compare_alphas(question, must_contain):
    dense_q = embeddings.embed_query(question)
    sparse_q = sparse_embed([question], "query")[0]
    rows = []
    for alpha in [1.0, 0.7, 0.5, 0.3, 0.0]:
        hits = search(dense_q, sparse_q, alpha, k=10)
        rank = next((i + 1 for i, h in enumerate(hits) if must_contain in h["text"]), None)
        rows.append({"alpha": alpha, "right chunk at rank": rank or "not in top 10",
                     "top result": f"{hits[0]['title']} p.{hits[0]['page']}"})
    print(question)
    display(pd.DataFrame(rows))

compare_alphas("What is SKU PRJ-PCB-PB1080 and how much does it cost?", "PRJ-PCB-PB1080")
compare_alphas("Can I work from home, and how many days do I need to be in the office?", "two days per week")

What is SKU PRJ-PCB-PB1080 and how much does it cost?


,alpha,right chunk at rank,top result
0,1.0,not in top 10,Product Catalog Electronics p.14
1,0.7,1,Product Catalog Electronics p.2
2,0.5,1,Product Catalog Electronics p.2
3,0.3,1,Product Catalog Electronics p.2
4,0.0,1,Product Catalog Electronics p.2


Can I work from home, and how many days do I need to be in the office?


,alpha,right chunk at rank,top result
0,1.0,1,Remote Work Policy p.2
1,0.7,2,Remote Work Policy p.3
2,0.5,2,Remote Work Policy p.3
3,0.3,2,Remote Work Policy p.3
4,0.0,2,Remote Work Policy p.3


<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 9 — Choose α automatically

A tiny rule (a mini *router*): if the question contains something that looks like a **code** (SKU, policy ID, model number such as `PRJ-PCB-PB1080`, `HR-REM-001`, `NQ-55U7`) → lean on **keywords** (α = 0.3); otherwise lean on **meaning** (α = 0.7).

</div>

In [8]:
CODE_PATTERN = re.compile(r"\b[A-Z0-9]{2,}(?:-[A-Z0-9]+)+\b")

def choose_alpha(question):
    return 0.3 if CODE_PATTERN.search(question) else 0.7

for q in ["What is SKU PRJ-PCB-PB1080?", "What does policy HR-REM-001 say?", "How do I request time off?"]:
    print(f"α = {choose_alpha(q)}  ←  {q}")

α = 0.3  ←  What is SKU PRJ-PCB-PB1080?
α = 0.3  ←  What does policy HR-REM-001 say?
α = 0.7  ←  How do I request time off?


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Step 10 — Multi-query + deduplication <span style="color:#757575;font-size:0.8em">(same as v1)</span>

Same idea as v1 (4 question versions → dedup by id and by text), but every search is now **hybrid** (🆕 dense + sparse for all 4 versions, two batched calls).

</div>

In [9]:
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class Rewrites(BaseModel):
    questions: list[str] = Field(description="3 alternative versions of the question")

rewrite_prompt = ChatPromptTemplate.from_template(
    "You help search IronStore's internal documents (HR, IT, finance, legal, sales, customer support, "
    "deliveries, product catalogs).\n"
    "Write 3 different versions of the employee's question. Use other words and synonyms an official "
    "policy or catalog would use. Keep product names, numbers and SKUs unchanged.\n\nQuestion: {question}"
)
rewriter = rewrite_prompt | ChatOpenAI(model=REWRITE_MODEL, temperature=0.3).with_structured_output(Rewrites)
MAX_CHUNKS = 8

def retrieve(question, k=6, verbose=False):
    queries = [question] + rewriter.invoke({"question": question}).questions[:3]
    alpha = choose_alpha(question)                                    # 🆕 v2
    dense_vecs = embeddings.embed_documents(queries)
    sparse_vecs = sparse_embed(queries, "query")                      # 🆕 v2
    with ThreadPoolExecutor(max_workers=len(queries)) as pool:
        results = list(pool.map(lambda pair: search(*pair, alpha, k), zip(dense_vecs, sparse_vecs)))

    raw = [{**hit, "query": f"Q{i}"} for i, hits in enumerate(results) for hit in hits]
    best = {}
    for hit in raw:                                                   # dedup by id
        if hit["id"] not in best:
            best[hit["id"]] = {**hit, "hits": 0}
        best[hit["id"]]["hits"] += 1
        best[hit["id"]]["score"] = max(best[hit["id"]]["score"], hit["score"])
    unique, seen_text = [], set()                                     # dedup by content
    for hit in sorted(best.values(), key=lambda h: (-h["hits"], -h["score"])):
        body = " ".join(hit["text"].split("\n", 1)[-1].split()).lower()
        if body not in seen_text:
            seen_text.add(body)
            unique.append(hit)
    unique = unique[:MAX_CHUNKS]

    if verbose:
        for i, q in enumerate(queries):
            print(f"Q{i}: {q}")
        print(f"\nα = {alpha} | {len(raw)} chunks found → {len(best)} after id dedup → {len(unique)} kept\n")
        overlap = pd.crosstab(pd.Series([h["id"] for h in raw], name="chunk"),
                              pd.Series([h["query"] for h in raw], name="found by"))
        display(overlap.replace({1: "✓", 0: ""}))
    return unique, alpha

kept, _ = retrieve("What is SKU PRJ-PCB-PB1080 and how much does it cost?", verbose=True)
pd.DataFrame(kept)[["hits", "score", "title", "page", "id"]]

Q0: What is SKU PRJ-PCB-PB1080 and how much does it cost?
Q1: Can you provide details on SKU PRJ-PCB-PB1080 and its pricing?
Q2: What information is available regarding SKU PRJ-PCB-PB1080 and its associated cost?
Q3: Could you clarify the specifications and price of SKU PRJ-PCB-PB1080?

α = 0.3 | 24 chunks found → 15 after id dedup → 8 kept



found by,Q0,Q1,Q2,Q3
chunk,,,,
IronStore_Discounts_and_Promotions_Policy-001,✓,,,
IronStore_Product_Catalog_Audio_and_Headphones-006,✓,,✓,
IronStore_Product_Catalog_Bags_and_Luggage-000,,✓,,
IronStore_Product_Catalog_Bags_and_Luggage-016,,,,✓
IronStore_Product_Catalog_Bags_and_Luggage-017,✓,✓,✓,✓
IronStore_Product_Catalog_Electronics-001,✓,✓,✓,✓
IronStore_Product_Catalog_Gaming_Accessories-022,,✓,,
IronStore_Product_Catalog_Health_and_Wellness-015,,,,✓
IronStore_Product_Catalog_Home_Gym_Equipment-000,,,✓,


,hits,score,title,page,id
0,4,9.530,Product Catalog Electronics,2,IronStore_Product_Catalog_Electronics-001
1,4,8.939,Product Catalog Bags and Luggage,37,IronStore_Product_Catalog_Bags_and_Luggage-017
2,2,6.837,Product Catalog Audio and Headphones,9,IronStore_Product_Catalog_Audio_and_Headphones...
3,2,6.479,Product Catalog Hunting Gear,29,IronStore_Product_Catalog_Hunting_Gear-020
4,2,5.183,Product Catalog Office Supplies,20,IronStore_Product_Catalog_Office_Supplies-015
5,1,7.758,Product Catalog Health and Wellness,18,IronStore_Product_Catalog_Health_and_Wellness-015
6,1,7.750,Product Catalog Mobile Phone Accessories,26,IronStore_Product_Catalog_Mobile_Phone_Accesso...
7,1,7.739,Product Catalog Toys and Games,13,IronStore_Product_Catalog_Toys_and_Games-010


<div style="background:#f3f3f3;border-left:6px solid #9e9e9e;padding:8px 14px;border-radius:4px">

### ♻️ Step 11 — Answer with sources <span style="color:#757575;font-size:0.8em">(same as v1)</span>

Same prompt and source links as v1; the footer now also shows α.

</div>

In [10]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the IronStore internal assistant for employees.\n"
     "Answer ONLY with information from the sources below. Cite them inline like [S1] or [S2][S3].\n"
     "If the sources do not contain the answer, say: \"I couldn't find this in the internal documents\" "
     "and suggest which department could help. Never invent numbers, names or rules.\n"
     "If sources disagree, prefer the most recently reviewed one and mention the difference.\n"
     "Sources marked OUTDATED (the 2021 catalog) must not be used for current prices or stock; "
     "if they are the only match, say the information is outdated.\n"
     "Be concise; use bullet points for procedures. Answer in the language of the question.\n\n"
     "Sources:\n{context}"),
    ("human", "{question}"),
])
answer_chain = answer_prompt | ChatOpenAI(model=ANSWER_MODEL, temperature=0)

def ask(question, verbose=False):
    t0 = time.time()
    kept, alpha = retrieve(question, verbose=verbose)
    t1 = time.time()
    context = "\n\n".join(f"[S{i}] {c['text']}" for i, c in enumerate(kept, 1))
    answer = answer_chain.invoke({"context": context, "question": question}).content
    t2 = time.time()

    cited = sorted({int(n) for n in re.findall(r"\[S(\d+)\]", answer)})
    lines = [f"- **[S{n}]** {kept[n-1]['title']}, page {kept[n-1]['page']}"
             f" — [open PDF]({Path(kept[n-1]['path']).as_uri()}#page={kept[n-1]['page']})"
             + (" ⚠️ outdated" if kept[n-1]["outdated"] else "")
             for n in cited if n <= len(kept)]
    display(Markdown(f"**Q: {question}**\n\n{answer}\n\n**Sources**\n" + ("\n".join(lines) or "- none")
                     + f"\n\n<sub>🆕 α = {alpha} · retrieval {t1 - t0:.1f} s · answer {t2 - t1:.1f} s</sub>"))
    return {"answer": answer, "sources": [kept[n-1] for n in cited if n <= len(kept)]}

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 12 — Tests: the v1 failures first

These two failed or were weak in v1 because they depend on **exact codes**:

</div>

In [12]:
_ = ask("how many days i can work from home in a week?")

**Q: how many days i can work from home in a week?**

- Most eligible hybrid employees are expected to attend their designated office an average of two days per week, implying they can work remotely the other days.
- Teams may require higher office attendance during peak periods or major projects.
- The specific working model (hybrid, remote-first, office-based) is confirmed by the hiring manager and People Partner and recorded in Workday.
- Remote-first employees normally work remotely except for required attendance at team events or business needs.
- Approval for remote or hybrid work requires a formal request and manager and People Partner review.
Therefore, typically you can work from home about three days per week under a hybrid model, but this depends on your team's arrangement and approval [S1][S2].

**Sources**
- **[S1]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)
- **[S2]** Remote Work Policy, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=1)

<sub>🆕 α = 0.7 · retrieval 2.8 s · answer 1.5 s</sub>

In [13]:
_ = ask("What is SKU TV-LMC-LC65M8?")

**Q: What is SKU TV-LMC-LC65M8?**

The SKU TV-LMC-LC65M8 corresponds to the Lumicore LC-65M8 Mini-LED Television. Key details include:

- Category: Electronics / Televisions
- Description: Large-format Mini-LED television with local dimming, gaming input support, and integrated voice control.
- Price: €1,199.00
- Specifications:
  - 65-inch 4K Mini-LED panel
  - Resolution: 3840 × 2160 pixels
  - 144 Hz variable refresh rate
  - HDR10+, Dolby Vision IQ, and HLG
  - 4 × HDMI 2.1, including eARC
  - AMD FreeSync Premium compatibility
  - 2.1-channel speakers, 60 W total output
  - VESA mount: 400 × 300 mm
- Availability: Backorder — 9 units expected 18 September 2026
- Warehouse location: Rotterdam DC, Oversize Bay O3 [S1]

**Sources**
- **[S1]** Product Catalog Electronics, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Product_Catalog_Electronics.pdf#page=2)

<sub>🆕 α = 0.3 · retrieval 2.8 s · answer 2.2 s</sub>

<span style="color:#757575">♻️ The v1 questions still work (meaning-based, α = 0.7):</span>

In [14]:
_ = ask("Can I work from home, and how many days do I need to be in the office?")

**Q: Can I work from home, and how many days do I need to be in the office?**

Yes, you can work from home if your role is eligible and you have approval following the remote work request process. The working models include:

- Hybrid: Work at an IronStore location on designated days and remotely on other days. Most hybrid employees are expected to attend the office about two days per week, though this may increase during peak periods or projects.
- Remote-first: Work is normally remote, with attendance required for team events, planning, training, or other business needs.

Approval requires submitting a request through Workday, manager assessment, People Partner review, and possibly country-specific checks. The arrangement is recorded in Workday and reviewed at least annually [S2][S7].

So, the number of office days depends on your approved working model and team requirements, typically around two days per week for hybrid roles [S2].

**Sources**
- **[S2]** Remote Work Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=2)
- **[S7]** Remote Work Policy, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Remote_Work_Policy.pdf#page=1)

<sub>🆕 α = 0.7 · retrieval 2.6 s · answer 1.7 s</sub>

In [15]:
_ = ask("How do I request time off, and how far in advance?")

**Q: How do I request time off, and how far in advance?**

To request time off at IronStore:

- Submit your annual leave request through PeopleHub.
- For leave of 1 to 5 working days, request at least 10 calendar days in advance.
- For leave exceeding 5 working days, request at least 20 calendar days in advance.
- For peak periods (e.g., July–August, year-end, major local holidays), request as early as possible.
- Do not book non-refundable travel until your leave is approved.
- Managers should respond within five working days, considering operational coverage fairly.

This process ensures proper planning and approval before your leave is confirmed [S3][S4].

**Sources**
- **[S3]** Employee Handbook, page 3 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Employee_Handbook.pdf#page=3)
- **[S4]** Leave & Absence Policy, page 2 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/internal_docs_by_area/HR_policies/IronStore_Leave_and_Absence_Policy.pdf#page=2)

<sub>🆕 α = 0.7 · retrieval 2.5 s · answer 1.8 s</sub>

In [16]:
_ = ask("What is the price of the IronStore 24 inch Full HD office monitor?")

**Q: What is the price of the IronStore 24 inch Full HD office monitor?**

The price of the IronStore 24" Full HD Office Monitor is €149.99 according to the 2021 catalog [S1]. However, this information is outdated and may not reflect the current price. For the latest pricing, please check with the Electronics department.

**Sources**
- **[S1]** Old Product Catalog 2021, page 1 — [open PDF](file:///C:/Users/Windows/Desktop/Ironhack/04_llm/project-brief-enterprise-ai-assistant/documents/extracted/documents/product_catalog/IronStore_Old_Product_Catalog_2021.pdf#page=1) ⚠️ outdated

<sub>🆕 α = 0.7 · retrieval 2.8 s · answer 1.3 s</sub>

In [17]:
_ = ask("What is the CEO's annual salary?")

**Q: What is the CEO's annual salary?**

I couldn't find this in the internal documents. You may contact the Finance or HR department for this information.

**Sources**
- none

<sub>🆕 α = 0.7 · retrieval 1.6 s · answer 0.8 s</sub>

<div style="background:#e8f5e9;border-left:6px solid #2e7d32;padding:10px 14px;border-radius:4px">

### 🆕 V2 — Step 13 — Findings

**✅ Better than v1**
- 🟢 **SKU questions now work.** `PRJ-PCB-PB1080` → PicoBeam projector, €399 (v1: "couldn't find"). `TV-LMC-LC65M8` → Lumicore TV, €1,199, backorder.
- 🟢 The α table shows why: with **α = 1 (v1 style)** the right chunk is **not even in the top 10**; with **any keyword share** it is **#1**.
- 🟢 Policy, outdated-catalog and "I don't know" answers stay as good as in v1.
- 🟢 Cost/speed: sparse vectors for all chunks took ~20 s; retrieval still ~2 s per question.

**⚠️ Still weak**
- 🟡 For meaning questions, hybrid moves the best chunk from #1 to #2 → keep α high (0.7) for those (done by `choose_alpha`).
- 🟡 Keyword search also matches common words (*sku, price, cost*) → for the SKU question 14 unique chunks came back, many unrelated. The LLM ignores them, but they cost tokens.
- 🟡 Hybrid scores are no longer cosine (e.g. 9.9), so a fixed "not found" threshold must be recalibrated.

**🚀 Ideas for v3**
1. **Reranker** (Pinecone `bge-reranker-v2-m3`) after dedup → keep only the truly relevant 3–5 chunks.
2. **Product-aware chunking** (one product per chunk + `sku` / `price` metadata) → exact SKU lookups by **metadata filter**, no search needed.
3. **Metadata routing**: product questions → catalogs only (skip `outdated`), policy questions → policies only.

</div>